In [1]:
# Cell 0 | 批量运行配置、模板路径和日志路径

import gc
import json
import os
import re
import sys
import traceback

from datetime import datetime
from pathlib import Path
from typing import Union

import torch


def find_project_root(
    start_path: Union[str, Path] = ".",
) -> Path:
    """向上查找同时包含data和notebooks的项目根目录。"""

    current_path = Path(start_path).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认Notebook位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


ROOT = find_project_root(
    Path.cwd()
)

EXPERIMENT_ROOT = (
    ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

FOLDWISE_OUTPUT_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "foldwise_mpnn_cv_features"
)

BATCH_LOG_DIR = (
    FOLDWISE_OUTPUT_ROOT
    / "batch_logs"
)

BATCH_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 已经验证通过的单折模板，批量版不会修改该文件
TEMPLATE_NOTEBOOK_PATH = (
    Path.cwd()
    / "MPNN特征预训练_粘度_折内无泄露版.ipynb"
)


SPLIT_ORDER = [
    "Data_split",
    "IL_split",
    "Cation_split",
    "Anion_split",
]


# 固定运行全部四种划分和全部五折。
# 已存在且audit_status=passed的折会自动跳过；
# 删除旧输出后重新运行时，将从头生成全部20折结果。
RUN_SPLITS = SPLIT_ORDER.copy()

RUN_FOLDS = list(
    range(
        1,
        6,
    )
)


# audit_status=passed的折自动跳过
SKIP_PASSED = True

# 某一折失败时立即停止，避免后续折在未知状态下继续运行
STOP_ON_ERROR = True


SESSION_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BATCH_RUN_STATE_PATH = (
    BATCH_LOG_DIR
    / f"batch_run_state_{SESSION_ID}.json"
)

LATEST_BATCH_RUN_STATE_PATH = (
    BATCH_LOG_DIR
    / "batch_run_state_latest.json"
)

BATCH_ERROR_PATH = (
    BATCH_LOG_DIR
    / f"batch_error_{SESSION_ID}.json"
)


if not TEMPLATE_NOTEBOOK_PATH.exists():
    raise FileNotFoundError(
        "没有找到已经验收通过的单折模板：\n"
        f"{TEMPLATE_NOTEBOOK_PATH}"
    )


invalid_splits = [
    split_name
    for split_name in RUN_SPLITS
    if split_name not in SPLIT_ORDER
]

if invalid_splits:
    raise ValueError(
        "RUN_SPLITS中存在无效划分："
        f"{invalid_splits}"
    )


invalid_folds = [
    fold
    for fold in RUN_FOLDS
    if fold not in [1, 2, 3, 4, 5]
]

if invalid_folds:
    raise ValueError(
        "RUN_FOLDS中存在无效折号："
        f"{invalid_folds}"
    )


planned_tasks = [
    (
        split_name,
        fold,
    )
    for split_name in RUN_SPLITS
    for fold in RUN_FOLDS
]


print("批量运行配置检查完成 ✓")
print("python                   =", sys.executable)
print(
    "device                   =",
    (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    ),
)
print("ROOT                     =", ROOT)
print("EXPERIMENT_ROOT          =", EXPERIMENT_ROOT)
print("template notebook        =", TEMPLATE_NOTEBOOK_PATH)
print("foldwise output root     =", FOLDWISE_OUTPUT_ROOT)
print("batch log directory      =", BATCH_LOG_DIR)
print("RUN_SPLITS               =", RUN_SPLITS)
print("RUN_FOLDS                =", RUN_FOLDS)
print("planned task count       =", len(planned_tasks))
print("SKIP_PASSED              =", SKIP_PASSED)
print("STOP_ON_ERROR            =", STOP_ON_ERROR)
print("external 10 points read  =", False)

批量运行配置检查完成 ✓
python                   = D:\conda\envs\rdkit-env\python.exe
device                   = cpu
ROOT                     = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
template notebook        = D:\jupyter\IL_ML_Project\notebooks\小论文_粘度预测_CO2筛选\01_10点留出验证\03_特征工程\MPNN特征预训练_粘度_折内无泄露版.ipynb
foldwise output root     = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features
batch log directory      = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\batch_logs
RUN_SPLITS               = ['Data_split', 'IL_split', 'Cation_split', 'Anion_split']
RUN_FOLDS                = [1, 2, 3, 4, 5]
planned task count       = 20
SKIP_PASSED              = True
STOP_ON_ERROR            = True
external 10 points read  = False


In [2]:
# Cell 1 | 读取并核对已经验收通过的单折模板

with open(
    TEMPLATE_NOTEBOOK_PATH,
    "r",
    encoding="utf-8",
) as file:
    template_notebook = json.load(
        file
    )


template_code_cells = [
    cell
    for cell in template_notebook["cells"]
    if (
        cell.get("cell_type") == "code"
        and "".join(
            cell.get(
                "source",
                [],
            )
        ).strip()
    )
]


expected_cell_titles = [
    "# Cell 0 | 导入依赖、设置折内MPNN运行路径和测试折",
    "# Cell 1 | 读取分子图、正式5折索引并核对外层训练测试集",
    "# Cell 2 | 在外层训练折内部建立MPNN训练集、验证集和标准化器",
    "# Cell 3 | 建立折内MPNN数据集和DataLoader",
    "# Cell 4 | 定义双分支MPNN编码器和黏度监督训练模型",
    "# Cell 5 | 单阶段折内训练MPNN并保存最佳模型与训练记录",
    "# Cell 6 | 计算折内MPNN指标并提取外层训练集和测试集特征",
    "# Cell 7 | 保存当前外层折的MPNN特征、索引、清单、指标和标准化器",
    "# Cell 8 | 从硬盘重新读取并审计Data_split-Fold 1全部输出",
]


if len(
    template_code_cells
) != len(
    expected_cell_titles
):
    raise RuntimeError(
        "单折模板中的非空代码Cell数量发生变化：\n"
        f"实际为{len(template_code_cells)}，"
        f"预期为{len(expected_cell_titles)}。\n"
        "请不要修改已经验收通过的单折模板。"
    )


template_cell_sources = []

for cell_number, (
    template_cell,
    expected_title,
) in enumerate(
    zip(
        template_code_cells,
        expected_cell_titles,
    )
):
    source = "".join(
        template_cell.get(
            "source",
            [],
        )
    )

    first_nonempty_line = next(
        (
            line.strip()
            for line in source.splitlines()
            if line.strip()
        ),
        "",
    )

    if first_nonempty_line != expected_title:
        raise RuntimeError(
            f"模板Cell {cell_number}标题不一致：\n"
            f"实际：{first_nonempty_line}\n"
            f"预期：{expected_title}"
        )

    template_cell_sources.append(
        source
    )


cell_0_source = template_cell_sources[0]


split_assignment_pattern = (
    r'(?m)^RUN_SPLIT_NAME = "Data_split"\s*$'
)

fold_assignment_pattern = (
    r"(?m)^RUN_FOLD = 1\s*$"
)


split_assignment_matches = re.findall(
    split_assignment_pattern,
    cell_0_source,
)

fold_assignment_matches = re.findall(
    fold_assignment_pattern,
    cell_0_source,
)


if len(
    split_assignment_matches
) != 1:
    raise RuntimeError(
        "模板Cell 0中的RUN_SPLIT_NAME赋值"
        "不是唯一的固定格式。"
    )


if len(
    fold_assignment_matches
) != 1:
    raise RuntimeError(
        "模板Cell 0中的RUN_FOLD赋值"
        "不是唯一的固定格式。"
    )


required_fixed_texts = [
    "Data_split - Fold 1",
    "Data_split Fold 1",
]

fixed_text_counts = {
    fixed_text: sum(
        source.count(
            fixed_text
        )
        for source in template_cell_sources
    )
    for fixed_text in required_fixed_texts
}


print("单折模板读取和结构核对通过 ✓")
print(
    "template code cell count =",
    len(template_code_cells),
)

print("\n模板Cell清单：")

for cell_number, source in enumerate(
    template_cell_sources
):
    title = next(
        (
            line.strip()
            for line in source.splitlines()
            if line.strip()
        ),
        "",
    )

    print(
        f"Cell {cell_number}: {title}"
    )


print("\n可替换运行参数：")
print(
    "RUN_SPLIT_NAME assignment count =",
    len(split_assignment_matches),
)
print(
    "RUN_FOLD assignment count       =",
    len(fold_assignment_matches),
)

print("\n固定显示文字数量：")

for fixed_text, count in (
    fixed_text_counts.items()
):
    print(
        f"{fixed_text!r} =",
        count,
    )

print(
    "\n模板当前执行输出不会被批量版使用，"
    "批量版只读取Cell源代码。"
)

单折模板读取和结构核对通过 ✓
template code cell count = 9

模板Cell清单：
Cell 0: # Cell 0 | 导入依赖、设置折内MPNN运行路径和测试折
Cell 1: # Cell 1 | 读取分子图、正式5折索引并核对外层训练测试集
Cell 2: # Cell 2 | 在外层训练折内部建立MPNN训练集、验证集和标准化器
Cell 3: # Cell 3 | 建立折内MPNN数据集和DataLoader
Cell 4: # Cell 4 | 定义双分支MPNN编码器和黏度监督训练模型
Cell 5: # Cell 5 | 单阶段折内训练MPNN并保存最佳模型与训练记录
Cell 6: # Cell 6 | 计算折内MPNN指标并提取外层训练集和测试集特征
Cell 7: # Cell 7 | 保存当前外层折的MPNN特征、索引、清单、指标和标准化器
Cell 8: # Cell 8 | 从硬盘重新读取并审计Data_split-Fold 1全部输出

可替换运行参数：
RUN_SPLIT_NAME assignment count = 1
RUN_FOLD assignment count       = 1

固定显示文字数量：
'Data_split - Fold 1' = 2
'Data_split Fold 1' = 1

模板当前执行输出不会被批量版使用，批量版只读取Cell源代码。


In [3]:
# Cell 2 | 批量运行四种划分全部折、自动跳过已完成折并保存运行状态

import csv
import gc
import time


LATEST_BATCH_ERROR_PATH = (
    BATCH_LOG_DIR
    / "batch_error_latest.json"
)

BATCH_SUMMARY_PATH = (
    BATCH_LOG_DIR
    / f"batch_fold_summary_{SESSION_ID}.csv"
)

LATEST_BATCH_SUMMARY_PATH = (
    BATCH_LOG_DIR
    / "batch_fold_summary_latest.csv"
)


def save_json_atomic(
    data,
    output_path: Path,
) -> None:
    """先写入临时文件，再替换正式JSON，避免中断时留下半个文件。"""

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_path,
        output_path,
    )


def save_summary_csv(
    records,
    output_path: Path,
) -> None:
    """保存每一折的批量运行摘要。"""

    summary_columns = [
        "split_name",
        "fold",
        "status",
        "audit_status",
        "elapsed_minutes",
        "outer_train_rows",
        "outer_test_rows",
        "mpnn_internal_train_rows",
        "mpnn_internal_validation_rows",
        "best_epoch",
        "best_validation_loss_standardized",
        "internal_validation_R2",
        "internal_validation_RMSE",
        "internal_validation_MAE",
        "internal_validation_Pearson_r",
        "started_at",
        "finished_at",
        "error_type",
        "error_message",
    ]

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=summary_columns,
            extrasaction="ignore",
        )

        writer.writeheader()

        for record in records:
            writer.writerow(
                {
                    column: record.get(
                        column,
                        "",
                    )
                    for column in summary_columns
                }
            )

    os.replace(
        temporary_path,
        output_path,
    )


def read_passed_audit(
    split_name: str,
    fold: int,
):
    """读取并核对某一折是否已经完成最终审计。"""

    audit_path = (
        FOLDWISE_OUTPUT_ROOT
        / split_name
        / f"Fold_{fold}"
        / "fold_output_audit.json"
    )

    if not audit_path.exists():
        return None

    try:
        with open(
            audit_path,
            "r",
            encoding="utf-8",
        ) as file:
            audit = json.load(file)

    except (
        json.JSONDecodeError,
        OSError,
    ):
        return None

    if audit.get(
        "audit_status"
    ) != "passed":
        return None

    if audit.get(
        "split_name"
    ) != split_name:
        return None

    if int(
        audit.get(
            "fold",
            -1,
        )
    ) != int(fold):
        return None

    return audit


def collect_fold_record(
    split_name: str,
    fold: int,
    status: str,
    started_at: str,
    finished_at: str,
    elapsed_seconds: float,
):
    """读取当前折审计文件和metadata，整理为一行批量摘要。"""

    fold_output_dir = (
        FOLDWISE_OUTPUT_ROOT
        / split_name
        / f"Fold_{fold}"
    )

    audit_path = (
        fold_output_dir
        / "fold_output_audit.json"
    )

    metadata_path = (
        fold_output_dir
        / "fold_feature_metadata.json"
    )

    with open(
        audit_path,
        "r",
        encoding="utf-8",
    ) as file:
        audit = json.load(file)

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as file:
        metadata = json.load(file)

    if audit.get(
        "audit_status"
    ) != "passed":
        raise RuntimeError(
            f"{split_name} Fold {fold}"
            "的最终审计状态不是passed。"
        )

    if audit.get(
        "split_name"
    ) != split_name:
        raise RuntimeError(
            f"{split_name} Fold {fold}"
            "审计文件中的划分名称错误。"
        )

    if int(
        audit.get(
            "fold",
            -1,
        )
    ) != int(fold):
        raise RuntimeError(
            f"{split_name} Fold {fold}"
            "审计文件中的折号错误。"
        )

    internal_validation_record = next(
        (
            record
            for record in metadata.get(
                "internal_metrics",
                [],
            )
            if record.get(
                "evaluation_subset"
            ) == "mpnn_internal_validation"
        ),
        None,
    )

    if internal_validation_record is None:
        raise RuntimeError(
            f"{split_name} Fold {fold}"
            "缺少MPNN内部验证指标。"
        )

    data_counts = audit.get(
        "data_counts",
        {},
    )

    training_checks = audit.get(
        "training_checks",
        {},
    )

    record = {
        "split_name": split_name,
        "fold": int(fold),
        "status": status,
        "audit_status": audit.get(
            "audit_status"
        ),
        "elapsed_minutes": (
            float(elapsed_seconds) / 60.0
        ),
        "outer_train_rows": int(
            data_counts[
                "outer_train_rows"
            ]
        ),
        "outer_test_rows": int(
            data_counts[
                "outer_test_rows"
            ]
        ),
        "mpnn_internal_train_rows": int(
            data_counts[
                "mpnn_internal_train_rows"
            ]
        ),
        "mpnn_internal_validation_rows": int(
            data_counts[
                "mpnn_internal_validation_rows"
            ]
        ),
        "best_epoch": int(
            training_checks[
                "best_epoch"
            ]
        ),
        "best_validation_loss_standardized": float(
            training_checks[
                "best_validation_loss_standardized"
            ]
        ),
        "internal_validation_R2": float(
            internal_validation_record[
                "R2"
            ]
        ),
        "internal_validation_RMSE": float(
            internal_validation_record[
                "RMSE"
            ]
        ),
        "internal_validation_MAE": float(
            internal_validation_record[
                "MAE"
            ]
        ),
        "internal_validation_Pearson_r": float(
            internal_validation_record[
                "Pearson_r"
            ]
        ),
        "started_at": started_at,
        "finished_at": finished_at,
        "error_type": "",
        "error_message": "",
    }

    return record


def update_batch_state(
    batch_state,
) -> None:
    """更新带时间戳的状态文件、latest状态文件和CSV摘要。"""

    records = batch_state[
        "records"
    ]

    batch_state[
        "updated_at"
    ] = datetime.now().isoformat(
        timespec="seconds"
    )

    batch_state[
        "summary"
    ] = {
        "planned": int(
            len(planned_tasks)
        ),
        "completed": int(
            sum(
                record.get(
                    "status"
                ) == "completed"
                for record in records
            )
        ),
        "skipped_passed": int(
            sum(
                record.get(
                    "status"
                ) == "skipped_passed"
                for record in records
            )
        ),
        "failed": int(
            sum(
                record.get(
                    "status"
                ) == "failed"
                for record in records
            )
        ),
        "processed": int(
            len(records)
        ),
    }

    save_json_atomic(
        batch_state,
        BATCH_RUN_STATE_PATH,
    )

    save_json_atomic(
        batch_state,
        LATEST_BATCH_RUN_STATE_PATH,
    )

    save_summary_csv(
        records,
        BATCH_SUMMARY_PATH,
    )

    save_summary_csv(
        records,
        LATEST_BATCH_SUMMARY_PATH,
    )


def build_fold_cell_sources(
    split_name: str,
    fold: int,
):
    """将已验收单折模板改写为当前划分、当前折的执行代码。"""

    patched_sources = []

    for (
        cell_number,
        original_source,
    ) in enumerate(
        template_cell_sources
    ):
        patched_source = str(
            original_source
        )

        if cell_number == 0:
            patched_source, split_count = (
                re.subn(
                    split_assignment_pattern,
                    (
                        "RUN_SPLIT_NAME = "
                        f"\"{split_name}\""
                    ),
                    patched_source,
                )
            )

            patched_source, fold_count = (
                re.subn(
                    fold_assignment_pattern,
                    f"RUN_FOLD = {int(fold)}",
                    patched_source,
                )
            )

            if split_count != 1:
                raise RuntimeError(
                    "批量替换RUN_SPLIT_NAME失败："
                    f"{split_name} Fold {fold}"
                )

            if fold_count != 1:
                raise RuntimeError(
                    "批量替换RUN_FOLD失败："
                    f"{split_name} Fold {fold}"
                )

        # 仅替换模板中写死的显示文字和审计说明，
        # 不改变实际索引、训练、特征提取与保存逻辑。
        patched_source = patched_source.replace(
            "Data_split - Fold 1",
            f"{split_name} - Fold {fold}",
        )

        patched_source = patched_source.replace(
            "Data_split Fold 1",
            f"{split_name} Fold {fold}",
        )

        patched_sources.append(
            patched_source
        )

    return patched_sources


batch_state = {
    "session_id": SESSION_ID,
    "status": "running",
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "updated_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "template_notebook": str(
        TEMPLATE_NOTEBOOK_PATH
    ),
    "foldwise_output_root": str(
        FOLDWISE_OUTPUT_ROOT
    ),
    "run_splits": list(
        RUN_SPLITS
    ),
    "run_folds": list(
        RUN_FOLDS
    ),
    "planned_tasks": [
        {
            "split_name": split_name,
            "fold": int(fold),
        }
        for split_name, fold in planned_tasks
    ],
    "skip_passed": bool(
        SKIP_PASSED
    ),
    "stop_on_error": bool(
        STOP_ON_ERROR
    ),
    "external_holdout_used": False,
    "records": [],
    "summary": {},
}

update_batch_state(
    batch_state
)


print("=" * 96)
print("开始批量运行折内无泄露MPNN特征提取")
print("=" * 96)
print("计划任务数 =", len(planned_tasks))
print("当前设备   =", (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
))
print("已通过审计的折将自动跳过。")
print()


for task_number, (
    split_name,
    fold,
) in enumerate(
    planned_tasks,
    start=1,
):
    fold_label = (
        f"{split_name} - Fold {fold}"
    )

    passed_audit = read_passed_audit(
        split_name=split_name,
        fold=fold,
    )

    if (
        SKIP_PASSED
        and passed_audit is not None
    ):
        now_text = datetime.now().isoformat(
            timespec="seconds"
        )

        skipped_record = collect_fold_record(
            split_name=split_name,
            fold=fold,
            status="skipped_passed",
            started_at=now_text,
            finished_at=now_text,
            elapsed_seconds=0.0,
        )

        batch_state[
            "records"
        ].append(
            skipped_record
        )

        update_batch_state(
            batch_state
        )

        print(
            f"[{task_number}/{len(planned_tasks)}] "
            f"{fold_label}："
            "已有passed审计，自动跳过 ✓"
        )

        continue


    print()
    print("=" * 96)
    print(
        f"[{task_number}/{len(planned_tasks)}] "
        f"开始运行 {fold_label}"
    )
    print("=" * 96)

    task_started_at = (
        datetime.now().isoformat(
            timespec="seconds"
        )
    )

    task_start_time = (
        time.perf_counter()
    )

    fold_namespace = {
        "__name__": "__main__",
    }

    try:
        fold_sources = (
            build_fold_cell_sources(
                split_name=split_name,
                fold=fold,
            )
        )

        for (
            template_cell_number,
            cell_source,
        ) in enumerate(
            fold_sources
        ):
            print()
            print(
                f">>> 正在执行模板Cell "
                f"{template_cell_number} | "
                f"{fold_label}"
            )

            code_object = compile(
                cell_source,
                (
                    f"{TEMPLATE_NOTEBOOK_PATH.name}"
                    f"::{split_name}"
                    f"::Fold_{fold}"
                    f"::Cell_{template_cell_number}"
                ),
                "exec",
            )

            exec(
                code_object,
                fold_namespace,
                fold_namespace,
            )

        task_end_time = (
            time.perf_counter()
        )

        task_finished_at = (
            datetime.now().isoformat(
                timespec="seconds"
            )
        )

        completed_record = (
            collect_fold_record(
                split_name=split_name,
                fold=fold,
                status="completed",
                started_at=task_started_at,
                finished_at=task_finished_at,
                elapsed_seconds=(
                    task_end_time
                    - task_start_time
                ),
            )
        )

        batch_state[
            "records"
        ].append(
            completed_record
        )

        update_batch_state(
            batch_state
        )

        print()
        print(
            f"{fold_label}运行和最终审计完成 ✓"
        )
        print(
            "本折耗时/分钟 =",
            completed_record[
                "elapsed_minutes"
            ],
        )
        print(
            "内部验证R2    =",
            completed_record[
                "internal_validation_R2"
            ],
        )
        print(
            "审计状态      =",
            completed_record[
                "audit_status"
            ],
        )

    except BaseException as error:
        task_end_time = (
            time.perf_counter()
        )

        task_finished_at = (
            datetime.now().isoformat(
                timespec="seconds"
            )
        )

        error_record = {
            "split_name": split_name,
            "fold": int(fold),
            "status": "failed",
            "audit_status": "",
            "elapsed_minutes": (
                float(
                    task_end_time
                    - task_start_time
                )
                / 60.0
            ),
            "outer_train_rows": "",
            "outer_test_rows": "",
            "mpnn_internal_train_rows": "",
            "mpnn_internal_validation_rows": "",
            "best_epoch": "",
            "best_validation_loss_standardized": "",
            "internal_validation_R2": "",
            "internal_validation_RMSE": "",
            "internal_validation_MAE": "",
            "internal_validation_Pearson_r": "",
            "started_at": task_started_at,
            "finished_at": task_finished_at,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }

        batch_state[
            "records"
        ].append(
            error_record
        )

        batch_state[
            "status"
        ] = "failed"

        batch_state[
            "failed_task"
        ] = {
            "split_name": split_name,
            "fold": int(fold),
        }

        update_batch_state(
            batch_state
        )

        batch_error = {
            "session_id": SESSION_ID,
            "created_at": task_finished_at,
            "split_name": split_name,
            "fold": int(fold),
            "error_type": type(error).__name__,
            "error_message": str(error),
            "traceback": traceback.format_exc(),
            "batch_run_state": str(
                BATCH_RUN_STATE_PATH
            ),
            "latest_batch_run_state": str(
                LATEST_BATCH_RUN_STATE_PATH
            ),
        }

        save_json_atomic(
            batch_error,
            BATCH_ERROR_PATH,
        )

        save_json_atomic(
            batch_error,
            LATEST_BATCH_ERROR_PATH,
        )

        print()
        print("=" * 96)
        print(
            f"{fold_label}运行失败 ✗"
        )
        print("=" * 96)
        print(
            "error type    =",
            type(error).__name__,
        )
        print(
            "error message =",
            str(error),
        )
        print(
            "saved error   =",
            BATCH_ERROR_PATH,
        )
        print(
            "latest error  =",
            LATEST_BATCH_ERROR_PATH,
        )

        if STOP_ON_ERROR:
            raise

    finally:
        fold_namespace.clear()

        del fold_namespace

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


if not any(
    record.get(
        "status"
    ) == "failed"
    for record in batch_state[
        "records"
    ]
):
    batch_state[
        "status"
    ] = "completed"

update_batch_state(
    batch_state
)


print()
print("=" * 96)
print("本轮批量运行结束")
print("=" * 96)
print(
    "batch status       =",
    batch_state[
        "status"
    ],
)
print(
    "planned tasks      =",
    batch_state[
        "summary"
    ][
        "planned"
    ],
)
print(
    "completed tasks    =",
    batch_state[
        "summary"
    ][
        "completed"
    ],
)
print(
    "skipped passed     =",
    batch_state[
        "summary"
    ][
        "skipped_passed"
    ],
)
print(
    "failed tasks       =",
    batch_state[
        "summary"
    ][
        "failed"
    ],
)

print("\n各折摘要：")

for record in batch_state[
    "records"
]:
    print(
        record[
            "split_name"
        ],
        f"Fold {record['fold']}",
        "| status =",
        record[
            "status"
        ],
        "| audit =",
        record[
            "audit_status"
        ],
        "| internal val R2 =",
        record.get(
            "internal_validation_R2",
            "",
        ),
        "| minutes =",
        record.get(
            "elapsed_minutes",
            "",
        ),
    )

print("\n批量日志文件：")
print(
    BATCH_RUN_STATE_PATH
)
print(
    LATEST_BATCH_RUN_STATE_PATH
)
print(
    BATCH_SUMMARY_PATH
)
print(
    LATEST_BATCH_SUMMARY_PATH
)

开始批量运行折内无泄露MPNN特征提取
计划任务数 = 20
当前设备   = cpu
已通过审计的折将自动跳过。

[1/20] Data_split - Fold 1：已有passed审计，自动跳过 ✓
[2/20] Data_split - Fold 2：已有passed审计，自动跳过 ✓
[3/20] Data_split - Fold 3：已有passed审计，自动跳过 ✓
[4/20] Data_split - Fold 4：已有passed审计，自动跳过 ✓
[5/20] Data_split - Fold 5：已有passed审计，自动跳过 ✓

[6/20] 开始运行 IL_split - Fold 1

>>> 正在执行模板Cell 0 | IL_split - Fold 1
python                   = D:\conda\envs\rdkit-env\python.exe
device                   = cpu
ROOT                     = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
RUN_SPLIT_NAME           = IL_split
RUN_FOLD                 = 1
FOLD_SEED                = 143
RUN_OUTPUT_DIR           = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\foldwise_mpnn_cv_features\IL_split\Fold_1
external 10 points read  = False

>>> 正在执行模板Cell 1 | IL_split - Fold 1
外层正式折读取和核对通过 ✓
all rows                = 16216
split                   = IL_split
fold         